In [2]:
import pandas as pd

# Load the two metadata files (real songs and fake songs)
fake = pd.read_csv("data/fake_songs.csv", low_memory=False)

real = pd.read_csv("data/real_songs.csv", low_memory=False)

print("Fake (AI) songs:", fake.shape)
print("Real (human) songs:", real.shape)

Fake (AI) songs: (49074, 17)
Real (human) songs: (48090, 13)


In [3]:
# Take a look at all the colummns for both
print("Fake (AI) columns:")
print(list(fake.columns))

print("\nReal (human) columns:")
print(list(real.columns))

Fake (AI) columns:
['id', 'lyrics', 'algorithm', 'style', 'filename', 'bit_rate', 'duration', 'source', 'lyrics_feature', 'topic', 'genre', 'mood', 'label', 'target', 'skip_time', 'no_vocal', 'split']

Real (human) columns:
['filename', 'title', 'artist', 'year', 'lyrics', 'duration', 'youtube_id', 'label', 'artist_overlap', 'target', 'skip_time', 'no_vocal', 'split']


In [4]:
from IPython.display import display

# Before I build the subset and start downloading audio I want to do a
# sanity check with the metadata to confirm what the columns I'll use actually are.
# Also to see what the unclear named columns are for (rather
# than guessing from their names).

# The columns I'll rely on later (filename, target, duration, algorithm,
# youtube_id) 
# Also the ones still unsure about (style, source,
# lyrics_feature, skip_time, artist_overlap):
fake_view = ["filename", "target", "duration", "algorithm",
             "style", "source", "lyrics_feature", "skip_time", "no_vocal"]

real_view = ["filename", "target", "duration", "youtube_id",
             "artist", "artist_overlap", "skip_time", "no_vocal"]


# Show the first 5 rows of just those columns as a readable table:
print("Fake (AI) — first 5 rows:")
display(fake[fake_view].head())

print("\nReal (human) — first 5 rows:")
display(real[real_view].head())

Fake (AI) — first 5 rows:


,filename,target,duration,algorithm,style,source,lyrics_feature,skip_time,no_vocal
0,fake_54113_suno_0,1,239.076,chirp-v3.5,"male vocals, lounge, piano, saxophone, slow te...",suno,* subject_matter: “The song reflects on the jo...,5.684094,False
1,fake_53851_suno_0,1,207.360,chirp-v3.5,"female vocals, salsa, piano, trumpet, percussi...",suno,* subject_matter: “The song tells the story of...,6.865344,False
2,fake_53851_suno_1,1,230.184,chirp-v3.5,"female vocals, salsa, piano, trumpet, percussi...",suno,* subject_matter: “The song tells the story of...,15.674094,False
3,fake_53853_suno_0,1,120.096,chirp-v3,"male vocals, guitar, drums, bass, grunge, mode...",suno,* subject_matter: “The song is about a person'...,0.030969,False
4,fake_53853_suno_1,1,120.096,chirp-v3,"male vocals, guitar, drums, bass, grunge, mode...",suno,* subject_matter: “The song is about a person'...,3.220344,False



Real (human) — first 5 rows:


,filename,target,duration,youtube_id,artist,artist_overlap,skip_time,no_vocal
0,real_00000,0,240.084,EZ-yNVc9Htw,Eminem,NaN,2.545344,False
1,real_00001,0,240.084,JGwWNGJdvx8,Ed Sheeran,NaN,15.876594,False
2,real_00002,0,177.084,ov4WobPqoSA,Kendrick Lamar,NaN,4.047219,False
3,real_00003,0,240.084,G5XpJP7f_SE,The Weeknd,NaN,3.524094,False
4,real_00004,0,240.084,_nxrYwT0SIo,Queen,True,4.857219,False


In [5]:
# Build a balanced subset with 2,000 AI and 2,000 human tracks
# A fixed random_state means we get the same sample every time it runs
# which keeps the project reproducible (NFR-002).

N = 2000
SEED = 42

fake_sample = fake.sample(n=N, random_state = SEED)

real_sample = real.sample(n=N, random_state = SEED)

print("AI sample:", fake_sample.shape)

print("Human sample:", real_sample.shape)


AI sample: (2000, 17)
Human sample: (2000, 13)


In [6]:
# Combine the two samples into one labelled dataset (4000 tracks)
subset = pd.concat([fake_sample, real_sample], ignore_index = True)

print("Combined subset:", subset.shape)

print(subset["target"].value_counts())

Combined subset: (4000, 22)
target
1    2000
0    2000
Name: count, dtype: int64


In [7]:
# Check the grouping with the artist aware split will use to prevent any leakage
# (Reals grouped by artist. Fakes by source song id)

print("Real (human) sample:")

print("unique artists:", real_sample["artist"].nunique())

print("most songs by one artist:")

print(real_sample["artist"].value_counts().head())

# fakes
print("\nFake (AI) sample:")

print("unique source ids:", fake_sample["id"].nunique())

print("most generations from one id:")

print(fake_sample["id"].value_counts().head())


Real (human) sample:
unique artists: 1468
most songs by one artist:
artist
Eminem                  7
The Weeknd              6
Kevin Gates             6
Florida Georgia Line    6
Panic! at the Disco     5
Name: count, dtype: int64

Fake (AI) sample:
unique source ids: 1963
most generations from one id:
id
17763    2
2955     2
31669    2
12576    2
16843    2
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import GroupShuffleSplit

# This will split it at 80/10/10 while keeping whole groups together
# no group is ever spread across two splits to avoid data leakage
def group_split_80_10_10(df, group_col, seed = 42):

    # 1) Take 80% as training and 20% as a temporary leftover split by group
    gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state = seed)

    train_idx, temp_idx = next(gss.split(df, groups = df[group_col]))

    train = df.iloc[train_idx]

    temp  = df.iloc[temp_idx]

    # 2) Half that leftover 20% into validation and test set roughyly 10%/10% overall
    gss2 = GroupShuffleSplit(n_splits = 1, train_size = 0.5, random_state = seed)

    val_idx, test_idx = next(gss2.split(temp, groups = temp[group_col]))

    val  = temp.iloc[val_idx]

    test = temp.iloc[test_idx]

    return train, val, test

# Apply to the human tracks grouping by artist
real_train, real_val, real_test = group_split_80_10_10(real_sample, "artist", seed = SEED)

print("Human tracks:")

print("train:", real_train.shape[0])

print("val:  ", real_val.shape[0])

print("test: ", real_test.shape[0])

Human tracks:
train: 1604
val:   204
test:  192


In [9]:
# using the the same function as above to the AI tracks grouping them by source song id
fake_train, fake_val, fake_test = group_split_80_10_10(fake_sample, "id", seed = SEED)

print("AI tracks:")

print("train:", fake_train.shape[0])

print("val:  ", fake_val.shape[0])

print("test: ", fake_test.shape[0])

AI tracks:
train: 1599
val:   199
test:  202


In [10]:
# Combine the AI and human splits so that each set has both classes
train = pd.concat([fake_train, real_train], ignore_index = True)

val   = pd.concat([fake_val,   real_val],   ignore_index = True)

test  = pd.concat([fake_test,  real_test],  ignore_index = True)

print("train:", train.shape)

print(train["target"].value_counts())

print("\nval:", val.shape)

print(val["target"].value_counts())

print("\ntest:", test.shape)

print(test["target"].value_counts())

train: (3203, 22)
target
0    1604
1    1599
Name: count, dtype: int64

val: (403, 22)
target
0    204
1    199
Name: count, dtype: int64

test: (394, 22)
target
1    202
0    192
Name: count, dtype: int64


In [11]:
# now need to verify that the split is leakage free so creating function
# so no artist (reals) and no source song id (fakes) appears in more than one split

def shared_groups(a, b, col):

    # this is to see if any group values appear in both dataframes
    return len(set(a[col]) & set(b[col]))

print("Real artists shared between splits (want 0 everywhere):")

print("train & val: ", shared_groups(real_train, real_val, "artist"))

print("train & test:", shared_groups(real_train, real_test, "artist"))

print("val & test:  ", shared_groups(real_val,   real_test, "artist"))


print("\nFake source ids shared between splits (want 0 everywhere):")

print("train & val: ", shared_groups(fake_train, fake_val, "id"))

print("train & test:", shared_groups(fake_train, fake_test, "id"))

print("val & test:  ", shared_groups(fake_val, fake_test, "id"))

Real artists shared between splits (want 0 everywhere):
train & val:  0
train & test: 0
val & test:   0

Fake source ids shared between splits (want 0 everywhere):
train & val:  0
train & test: 0
val & test:   0


In [12]:
# now need to save it to disk so I don't have to rerun it each time
import os

# Label each set with its split name
# (this overwrites the dataset's own 'split' column, which we're not using)
train["split"] = "train"
val["split"]   = "val"
test["split"]  = "test"

# Stack the three back into one table now carrying a split label per row
splits = pd.concat([train, val, test], ignore_index = True)

# Write it out as a CSV file
os.makedirs("data", exist_ok = True)

splits.to_csv("data/subset_splits.csv", index = False)

print("Saved:", splits.shape, "to data/subset_splits.csv")

print(splits["split"].value_counts())

Saved: (4000, 22) to data/subset_splits.csv
split
train    3203
val       403
test      394
Name: count, dtype: int64
